In [1]:

import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from data.queries import engine, Session
from data.models import Paper
from typing import Optional
from sqlalchemy import func


def get_study_by(attr: str, value: int | str, after_2025: bool=False) -> dict:
    session = Session()
    try:
        if after_2025:
            query = session.query(Paper).filter(
                Paper.entrez_year > 2025
            )
        else:
            query = session.query(Paper).filter(
                (Paper.entrez_year.is_(None)) | (Paper.entrez_year <= 2025)
            )
        query = query.filter(getattr(Paper, attr) == value)

        # if there is several papers raise an error
        papers = query.all()
        if len(papers) > 1:
            print(papers)
            
            # raise ValueError(f"Multiple papers found for {attr} = {value}")

        paper = query.first()
        if not paper:
            return {}
        else:
            return {
                "id": paper.id,
                "pubmed_id": paper.pubmed_id,
                "doi": paper.doi,
                "title": paper.title,
                "abstract": paper.abstract,
            }
    finally:
        session.close()


def get_study_by_title(title: str, include_substring: bool = False, after_2025: bool=False) -> list[dict]:
    title = title.lower()
    session = Session()
    try:
        if include_substring:
            query = session.query(Paper).filter(
                func.lower(Paper.title).contains(func.lower(title))
            )
        else:
            query = session.query(Paper).filter(
                func.lower(Paper.title) == func.lower(title)
            )
        if after_2025:
            query = query.filter(
                Paper.entrez_year > 2025
            )
        else:
            query = query.filter(
                (Paper.entrez_year.is_(None)) | (Paper.entrez_year <= 2025)
            )
        papers = query.all()

        if not papers:
            return []

        result = []
        for paper in papers:
            result.append({
                "id": paper.id,
                "pubmed_id": paper.pubmed_id,
                "doi": paper.doi,
                "title": paper.title,
                "abstract": paper.abstract,
            })

        return result

    finally:
        session.close()


def read_in_asreview() -> list[pd.DataFrame]:
    included = pd.read_csv('../data/manual/as_review_studies_relevant_with_info_20240101_00-00-00.csv')
    excluded = pd.read_csv('../data/manual/as_review_studies_excluded_with_pubmed_id.csv')
    excluded_after_stopping = pd.read_csv('../data/manual/as_review_studies_excluded_after_stopping_with_pubmed_id.csv')
   
    return [included, excluded, excluded_after_stopping]


def read_in_all_retrieved_studies() -> list[pd.DataFrame]:
    input_dir = '../data/relevant_studies/studies_20260530_02-04-42.csv'
    df = pd.read_csv(input_dir)
    # remove all papers that have entrez_date > 2025
    df['entrez_date'] = pd.to_datetime(df['entrez_year'], errors='coerce')
    df = df[df['entrez_date'] < '2025-01-01']

    included = df.loc[df['prediction'] == 1].copy()
    excluded = df.loc[df['prediction'] == 0].copy()
    # convert title to lowercase
    included.loc[:, 'title'] = included['title'].str.lower()
    excluded.loc[:, 'title'] = excluded['title'].str.lower()
    return [included, excluded]


def find_study_in_db(pmid: int, doi: str, title: str, after_2025: bool = False, rel_pmid1: Optional[int] = None, rel_pmid2: Optional[int] = None) -> bool:
    study_data = None
    if pmid:
        study_data = get_study_by('pubmed_id', pmid, after_2025=after_2025)
    if study_data:
        return True
    else:
        if rel_pmid1:
            study_data = get_study_by('pubmed_id', rel_pmid1, after_2025=after_2025)
        if study_data:
            return True
        if rel_pmid2:
            study_data = get_study_by('pubmed_id', rel_pmid2, after_2025=after_2025)
        if study_data:
            return True
        if doi:
            doi = str(doi).replace('https://doi.org/', '')
            study_data = get_study_by('doi', doi, after_2025=after_2025)
        if study_data:
            return True
        else:
            study_data = get_study_by_title(title, after_2025=after_2025)
            if study_data:
                return True
            else:
                return False        

In [2]:
articles = "../validation/psynamic_validation_included_articles.csv"
reviews = "../validation/psynamic_validation_sr_library.csv"

df_articles = pd.read_csv(articles, delimiter=';')
df_articles.replace('nA', None, inplace=True)

df_reviews = pd.read_csv(reviews, delimiter=';')
df_reviews.replace('nA', None, inplace=True)

included_man, excluded_man, excluded_after_stopping_man = read_in_asreview()
included_auto, excluded_auto = read_in_all_retrieved_studies()

How many articles and systematic review are there in the set?

In [3]:
print(len(df_articles))
print(len(df_reviews))

413
30


How many studies per review?

In [4]:
# mean of papers per article
df_articles.groupby("SR_No").size().mean()

# 


np.float64(13.766666666666667)

How many systematic reviews?

In [5]:
nr_studie_in_db = 0

df_articles['in_psynamic'] = False
df_articles['in_excluded'] = False
df_articles['in_excluded_after_stopping'] = False
df_articles['excluded_by_bert'] = False
df_articles['in_later_retrieved'] = False

for index, row in df_articles.iterrows():
    title = row['Title'].lower()
    if find_study_in_db(row['PMID'], row['doi'], row['Title'], after_2025=False, rel_pmid1=row['Related Publication PMID 1'], rel_pmid2=row['Related Publication PMID 2']):
        df_articles.at[index, 'in_psynamic'] = True

    elif title in excluded_man['title'].values:
        df_articles.at[index, 'in_excluded'] = True

    elif title in excluded_after_stopping_man['title'].values:
        df_articles.at[index, 'in_excluded_after_stopping'] = True

    if title in excluded_auto['title'].values:
        df_articles.at[index, 'excluded_by_bert'] = True

    if find_study_in_db(row['PMID'], row['doi'], row['Title'], after_2025=True, rel_pmid1=row['Related Publication PMID 1'], rel_pmid2=row['Related Publication PMID 2']):
        df_articles.at[index, 'in_later_retrieved'] = True

Number of articles excluded by BERT?

In [6]:
df_articles['excluded_by_bert'].sum()


np.int64(0)

How large is the overlap between the articles (with duplicates)?

In [7]:
nr_articles = df_articles['in_psynamic'].sum()
print(nr_articles)
percentage = (nr_articles / len(df_articles)) * 100
print(percentage)

368
89.10411622276028


Percentage of articles from RS that are included in PsyNamic database (up until 2025. no duplicates)?

In [8]:
df_articles_no_duplicates = df_articles.drop_duplicates(subset=['PMID', 'doi', 'Title'], keep='first')
print(len(df_articles_no_duplicates))
nr_articles_no_duplicate = df_articles_no_duplicates['in_psynamic'].sum()
print(nr_articles_no_duplicate)
print(nr_articles_no_duplicate / len(df_articles_no_duplicates) * 100)

322
281
87.26708074534162


In [9]:
# get number of articles where either in_psyanamic, in_excluded or in_excluded_after_stopping is True
df_articles_no_duplicates['in_psynamic_or_excluded'] = df_articles_no_duplicates['in_psynamic'] | df_articles_no_duplicates['in_excluded'] | df_articles_no_duplicates['in_excluded_after_stopping']
nr_articles_no_duplicate_or_excluded = df_articles_no_duplicates['in_psynamic_or_excluded'].sum()
print(nr_articles_no_duplicate_or_excluded)
(nr_articles_no_duplicate_or_excluded / len(df_articles_no_duplicates)) * 100

293


/tmp/ipykernel_277444/3562798738.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_articles_no_duplicates['in_psynamic_or_excluded'] = df_articles_no_duplicates['in_psynamic'] | df_articles_no_duplicates['in_excluded'] | df_articles_no_duplicates['in_excluded_after_stopping']


np.float64(90.99378881987577)

Number of studies from RS that were manually excluded?

In [10]:
print(df_articles_no_duplicates['in_excluded'].sum())
df_articles_no_duplicates[df_articles_no_duplicates['in_excluded'] == True][['SR_No','Title']]

7


,SR_No,Title
54,3,Psychedelic microdosing benefits and challenge...
65,3,Short-Term Treatment Effects of a Substance Us...
121,6,Time until relapse after augmentation with sin...
225,16,A preliminary investigation of ibogaine: case ...
240,16,"Ibogaine: Complex pharmacokinetics, concerns f..."
325,23,Short-Term Treatment Effects of a Substance Us...
331,24,"Persisting reductions in cannabis, opioid, and..."


Number of studies from RS that were in excluded after stopping criteria?

In [11]:
df_articles_no_duplicates['in_excluded_after_stopping'].sum()

np.int64(5)

Number of studies from RS that were included after 2025?

In [12]:
df_articles_no_duplicates['in_later_retrieved'].sum()

np.int64(0)

What is the average percentage of articles from RS that are included in PsyNamic database (up until 2025)?

In [13]:
sr_nos = df_articles['SR_No'].unique()
# add column overlap
df_reviews['overlap'] = pd.NA
df_reviews['in_psynamic'] = pd.NA
for sr_no in sr_nos:
    total_studies = df_reviews[df_reviews['SR'] == sr_no]['Total Nr'].iloc[0]
    included = df_articles[(df_articles['SR_No'] == sr_no) & (df_articles['in_psynamic'])].shape[0] 
    percentage = included / total_studies * 100 if total_studies > 0 else 0
    df_reviews.loc[df_reviews['SR'] == sr_no, 'overlap'] = percentage
    nr_included = df_articles[(df_articles['SR_No'] == sr_no) & (df_articles['in_psynamic'])].shape[0]
    df_reviews.loc[df_reviews['SR'] == sr_no, 'in_psynamic'] = nr_included
df_reviews[['SR', 'Authors', 'overlap', 'in_psynamic']]
# round overlap
df_reviews['overlap'] = pd.to_numeric(df_reviews['overlap']).round(1
                                                                   )
print(df_reviews['overlap'].to_list())
print(df_reviews['overlap'].mean())
print(df_reviews['overlap'].median())

print(df_reviews['in_psynamic'].mean())
print(df_reviews['in_psynamic'].median())

[100.0, 93.3, 59.3, 85.7, 58.3, 84.8, 20.0, 80.0, 56.7, 100.0, 100.0, 100.0, 80.0, 100.0, 100.0, 70.8, 100.0, 100.0, 100.0, 44.4, 91.7, 100.0, 75.0, 94.7, 100.0, 27.3, 88.9, 92.9, 66.7, 90.0]
82.01666666666667
90.85
12.266666666666667
9.0


In [14]:
print(f'Average overlap: {df_reviews["overlap"].mean():.2f}%')
print(f'Median overlap: {df_reviews["overlap"].median():.2f}%')


Average overlap: 82.02%
Median overlap: 90.85%


What's the median of in psynamic and in included?

In [15]:
df_reviews['in_psynamic'].median()

np.float64(9.0)

In [16]:
df_reviews['Included'].median()

np.float64(11.0)

Number of missing articles that don't have a PMID?

In [17]:
df_filtered = df_articles_no_duplicates[
    df_articles_no_duplicates['in_psynamic_or_excluded'] == False
]

# count where the PMID is None
nr_no_pmid = df_filtered['PMID'].isna().sum()
# percentage of articles without PMID
(nr_no_pmid / len(df_filtered)) * 100
nr_no_pmid

np.int64(13)

Random sample of 20 articles that were not found by search string

In [18]:
# check how many of the not included the pmid is none
df_articles_not_found_in_psynamic = df_articles[
    (df_articles['in_psynamic'] == False) &
    (df_articles['in_excluded'] == False) &
    (df_articles['in_excluded_after_stopping'] == False)
]
df_subset = df_articles_not_found_in_psynamic.sample(n=20, random_state=40)

# df_subset[['Title'], ['PMID'], ['doi']]
# write out csv with only these columns
df_subset[['Title', 'PMID', 'doi']].to_csv('/home/veral/PsyNamic/PsyNamic-Webapp/validation/20_random_sample_not_found_in_psynamic.csv', index=False)
